# cyberdef-vuln - defensive vulnerability detection (demo)

Paste code, get a `VERDICT` (VULNERABLE / SAFE). This is a blue-team **detection** assistant,
not an authority: a false SAFE is the dangerous failure mode, so keep a human reviewer.

- Model: https://huggingface.co/pallav0101/cyberdef-vuln-qwen2.5-7b-primevul
- Code & full results: https://github.com/pallav0107/cyberdef-llm

**First:** Runtime -> Change runtime type -> T4 GPU, then run the cells top to bottom.

In [ ]:
!pip -q install transformers peft accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER = "pallav0101/cyberdef-vuln-qwen2.5-7b-primevul"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

In [ ]:
SYS = ("You are a security code reviewer performing defensive vulnerability detection. "
       "Decide whether the snippet contains a security vulnerability. Answer with exactly "
       "'VERDICT: VULNERABLE' or 'VERDICT: SAFE' on the first line, then one short line naming the issue.")

def analyze(code):
    msgs = [{"role": "system", "content": SYS},
            {"role": "user", "content": "Code:\n" + code}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

print(analyze("import os\ndef run(c):\n    os.system('ping ' + c)  # user-controlled input"))

In [ ]:
# Try your own:
print(analyze("""import sqlite3
def get(db, name):
    return db.execute("SELECT * FROM users WHERE name = '" + name + "'").fetchall()"""))